# agent_framework_purview

## Example 1: settings and the Graph scopes they need

In [ ]:
# Example 1: PurviewSettings is a typed dictionary; get_purview_scopes derives the token scope from the Graph URI
from agent_framework_purview import PurviewSettings, get_purview_scopes

settings = PurviewSettings(app_name="smoke-app", app_version="1.0")
assert get_purview_scopes(settings) == ["https://graph.microsoft.com/.default"]
print("Example 1:", dict(settings), get_purview_scopes(settings))

## Example 2: a sovereign-cloud Graph endpoint changes the scope

In [ ]:
# Example 2: a different graph_base_uri gives that cloud's scope
from agent_framework_purview import PurviewSettings, get_purview_scopes

scopes = get_purview_scopes(PurviewSettings(app_name="gov", graph_base_uri="https://graph.microsoft.us/v1.0/"))
assert scopes and scopes[0].startswith("https://graph.microsoft.us"), scopes
print("Example 2:", scopes)

## Example 3: an app location and the blocked-message texts

In [ ]:
# Example 3: where the app runs (application / uri / domain) and what a blocked prompt or response says
from agent_framework_purview import PurviewAppLocation, PurviewLocationType, PurviewSettings

location = PurviewAppLocation(location_type=PurviewLocationType.APPLICATION, location_value="00000000-0000-0000-0000-000000000001")
settings = PurviewSettings(app_name="smoke", purview_app_location=location, blocked_prompt_message="Blocked by policy.")
assert settings["purview_app_location"].location_type is PurviewLocationType.APPLICATION
print("Example 3:", [t.value for t in PurviewLocationType], settings["blocked_prompt_message"])

## Example 4: the policy middleware, built with a static credential

In [ ]:
# Example 4: PurviewPolicyMiddleware wraps an agent's runs; building it sends nothing
from azure.core.credentials import AccessToken
from agent_framework_purview import PurviewChatPolicyMiddleware, PurviewPolicyMiddleware, PurviewSettings

class StaticToken:
    def get_token(self, *scopes, **kwargs):
        return AccessToken("test-token", 4102444800)

agent_mw = PurviewPolicyMiddleware(StaticToken(), PurviewSettings(app_name="smoke"))
chat_mw = PurviewChatPolicyMiddleware(StaticToken(), PurviewSettings(app_name="smoke"))
print("Example 4:", type(agent_mw).__name__, type(chat_mw).__name__)

## Example 5: `agent_framework.microsoft` re-exports this package

In [ ]:
# Example 5: the umbrella namespace for Microsoft connectors hands back the same classes
import importlib.metadata
import agent_framework.microsoft as via_core
import agent_framework_purview as direct

assert via_core.PurviewPolicyMiddleware is direct.PurviewPolicyMiddleware and via_core.PurviewSettings is direct.PurviewSettings
print("Example 5:", importlib.metadata.version("agent-framework-purview"), "re-exported through agent_framework.microsoft")